In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import PowerNorm
from scipy.ndimage import gaussian_filter1d
from scipy.stats import circmean, pearsonr, permutation_test, spearmanr

from Utils import rfmap
from Utils.json_tools import read_formatted_json
from Utils.plotting import apply_light_plot_style
from Utils.rf_trials import load_regular_rf_trials
from Utils.rfmap import plot_1d_rfmap

apply_light_plot_style()

In [ ]:
base_dir = Path("/mnt/senzailab/Kai/#Recording/m19")
date = 260824
hd_session_id = 1
rf_session_id = 2
probes = ("A", )

hd_bin_count = 30
hd_smoothing_deg = 3.0
rf_smoothing_bins = 1.5
rf_response_window_s = (0.0, 0.2)
rf_detection_permutations = 10_000
rf_detection_seed = 0
density_bin_count = 12
statistics_permutations = 10_000
statistics_seed = 1

session_dir = base_dir / str(date)
hd_session = session_dir / f"{date}_{hd_session_id}"
rf_session = session_dir / f"{date}_{rf_session_id}"
probe_rank = {probe: rank for rank, probe in enumerate(probes)}


def ordered_unit_keys(keys):
    return sorted(keys, key=lambda key: (probe_rank[key[0]], key[1]))

In [ ]:
# Detect RF cells independently from all RF-session units.
rf_unit_key_set = set()
rf_profile_by_key = {}
rf_peak_angle_by_key = {}
rf_angles_by_probe = {}

for probe in probes:
    rf_source_path = (
        rf_session
        / "data/rfmapping/good/-100_400_1ms"
        / f"Probe{probe}"
        / f"regular_unitsSpikeCounts_{date}_{rf_session_id}.rfmap"
    )
    summed_maps = rfmap.load_rf_maps(rf_source_path).sum(
        *rf_response_window_s,
        show_progress=False,
    )
    rf_masks = summed_maps.rf_2d(
        load_regular_rf_trials(
            rf_session,
            probe,
            summed_maps,
            on=True,
            off=False,
        ),
        is_shuffle=True,
        cluster_forming_z=1.5,
        alpha=0.05,
        n_permutations=rf_detection_permutations,
        random_seed=rf_detection_seed,
        wrap_x=True,
        show_progress=False,
    )

    raw_angles_deg = np.mod(-summed_maps[0].x_positions, 360.0)
    rf_plot_order = np.argsort(raw_angles_deg)
    rf_angles_by_probe[probe] = raw_angles_deg[rf_plot_order]
    profiles = summed_maps.to_1d_array(axis="x")[:, rf_plot_order]
    profiles = gaussian_filter1d(
        profiles.astype(float),
        sigma=rf_smoothing_bins,
        axis=1,
        mode="wrap",
    )

    for unit_map, rf_mask, profile in zip(
        summed_maps,
        rf_masks,
        profiles,
        strict=True,
    ):
        if not np.any(rf_mask):
            continue
        unit_key = (probe, int(unit_map.unit_id))
        rf_unit_key_set.add(unit_key)
        rf_profile_by_key[unit_key] = profile
        rf_peak_angle_by_key[unit_key] = float(
            rf_angles_by_probe[probe][np.argmax(profile)]
        )

rf_unit_keys = ordered_unit_keys(rf_unit_key_set)
rf_unit_ids = np.asarray([unit_id for _, unit_id in rf_unit_keys], dtype=int)
rf_unit_ids

In [ ]:
# Select HD cells independently and build their 30-bin tuning curves.
hd_angle_bin_edges = np.linspace(0.0, 360.0, hd_bin_count + 1)
hd_angle_bin_centers = (hd_angle_bin_edges[:-1] + hd_angle_bin_edges[1:]) / 2
hd_unit_key_set = set()
hd_rate_by_key = {}
hd_peak_angle_by_key = {}

for probe in probes:
    tuning_dir = hd_session / f"data/tuning_curves/Probe{probe}"
    preferred_path = tuning_dir / "tuning_curves.tc"
    legacy_path = tuning_dir / "tuning_curves.json"
    hd_path = preferred_path if preferred_path.is_file() else legacy_path
    hd_data = read_formatted_json(hd_path)

    unit_ids = np.asarray(hd_data["unit_id"], dtype=int)
    hd_classes = np.asarray(
        hd_data["unit_data"]["hd_class"],
        dtype=object,
    )
    if hd_classes.shape != unit_ids.shape:
        raise ValueError("HD classes must align with HD unit IDs")
    hd_rows = np.flatnonzero(hd_classes == 2)
    if hd_rows.size == 0:
        continue

    spike_counts = np.asarray(hd_data["spike_counts"], dtype=float)[hd_rows]
    occupancy = np.asarray(hd_data["occupancy_time_s"], dtype=float)
    source_bin_count = spike_counts.shape[1]
    if source_bin_count % hd_bin_count != 0:
        raise ValueError("HD source bins must divide evenly into hd_bin_count")

    bins_per_group = source_bin_count // hd_bin_count
    smoothing_sigma = hd_smoothing_deg / (360.0 / source_bin_count)
    smoothed_counts = gaussian_filter1d(
        spike_counts,
        sigma=smoothing_sigma,
        axis=1,
        mode="wrap",
    ).reshape(len(hd_rows), hd_bin_count, bins_per_group).sum(axis=2)
    smoothed_occupancy = gaussian_filter1d(
        occupancy,
        sigma=smoothing_sigma,
        mode="wrap",
    ).reshape(hd_bin_count, bins_per_group).sum(axis=1)
    probe_rates = np.divide(
        smoothed_counts,
        smoothed_occupancy[None, :],
        out=np.full(smoothed_counts.shape, np.nan),
        where=smoothed_occupancy[None, :] > 0,
    )

    for row, rate in zip(hd_rows, probe_rates, strict=True):
        unit_key = (probe, int(unit_ids[row]))
        hd_unit_key_set.add(unit_key)
        hd_rate_by_key[unit_key] = rate
        hd_peak_angle_by_key[unit_key] = float(
            hd_angle_bin_centers[np.nanargmax(rate)]
        )

hd_unit_keys = ordered_unit_keys(hd_unit_key_set)
hd_unit_ids = np.asarray([unit_id for _, unit_id in hd_unit_keys], dtype=int)
hd_unit_ids

In [ ]:
# Literal union is reported; paired analyses use the overlap/intersection.
union_unit_keys = ordered_unit_keys(rf_unit_key_set | hd_unit_key_set)
overlap_unit_keys = ordered_unit_keys(rf_unit_key_set & hd_unit_key_set)
union_unit_ids = np.asarray([unit_id for _, unit_id in union_unit_keys], dtype=int)
overlap_unit_ids = np.asarray(
    [unit_id for _, unit_id in overlap_unit_keys],
    dtype=int,
)


def hd_angle_to_rf_layout_x(angle_deg):
    angle_deg = np.asarray(angle_deg, dtype=float)
    return -((angle_deg + 180.0) % 360.0 - 180.0)


rf_layout_x_by_hd_bin = hd_angle_to_rf_layout_x(hd_angle_bin_centers)
rf_layout_column_order = np.argsort(rf_layout_x_by_hd_bin)


def sorted_by_angle(unit_keys, angle_by_key):
    # Previous: sort monotonically from 0 to 360 degrees.
    # return sorted(
    #     unit_keys,
    #     key=lambda key: (angle_by_key[key], probe_rank[key[0]], key[1]),
    # )
    return sorted(
        unit_keys,
        key=lambda key: (
            float(hd_angle_to_rf_layout_x(angle_by_key[key])),
            probe_rank[key[0]],
            key[1],
        ),
    )


rf_sort_sequence = sorted_by_angle(rf_unit_keys, rf_peak_angle_by_key)
hd_sort_sequence = sorted_by_angle(hd_unit_keys, hd_peak_angle_by_key)
overlap_rf_sort_sequence = sorted_by_angle(
    overlap_unit_keys,
    rf_peak_angle_by_key,
)
overlap_hd_sort_sequence = sorted_by_angle(
    overlap_unit_keys,
    hd_peak_angle_by_key,
)

rf_sort_unit_ids = np.asarray([key[1] for key in rf_sort_sequence], dtype=int)
hd_sort_unit_ids = np.asarray([key[1] for key in hd_sort_sequence], dtype=int)
overlap_rf_sort_unit_ids = np.asarray(
    [key[1] for key in overlap_rf_sort_sequence],
    dtype=int,
)
overlap_hd_sort_unit_ids = np.asarray(
    [key[1] for key in overlap_hd_sort_sequence],
    dtype=int,
)

assert set(overlap_rf_sort_sequence) == set(overlap_hd_sort_sequence)
assert all(key in rf_profile_by_key for key in overlap_unit_keys)
assert all(key in hd_rate_by_key for key in overlap_unit_keys)

cell_unit_ids = {
    "rf": rf_unit_ids,
    "hd": hd_unit_ids,
    "union": union_unit_ids,
    "overlap": overlap_unit_ids,
}
sorting_unit_ids = {
    "rf_peak": rf_sort_unit_ids,
    "hd_preferred_direction": hd_sort_unit_ids,
    "overlap_rf_peak": overlap_rf_sort_unit_ids,
    "overlap_hd_preferred_direction": overlap_hd_sort_unit_ids,
}
{
    "groups": {name: values.tolist() for name, values in cell_unit_ids.items()},
    "sorts": {name: values.tolist() for name, values in sorting_unit_ids.items()},
}

## Six requested heatmaps

In [ ]:
def plot_keyed_heatmap(row_by_key, unit_key_sequence):
    # Previous: display columns monotonically from 0 to 360 degrees.
    # plot_1d_rfmap(
    #     np.stack([row_by_key[key] for key in unit_key_sequence]),
    #     label_list=[
    #         f"{probe}:{unit_id}" for probe, unit_id in unit_key_sequence
    #     ],
    #     isHeatmap=True,
    #     xinDeg=True,
    #     isNormalize=True,
    # )
    data = np.stack([row_by_key[key] for key in unit_key_sequence])
    data = data[:, rf_layout_column_order].astype(float)
    row_max = np.nanmax(data, axis=1, keepdims=True)
    data = np.divide(
        data,
        row_max,
        out=np.zeros_like(data),
        where=np.isfinite(row_max) & (row_max != 0),
    )

    n_units = data.shape[0]
    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(
        data,
        aspect="auto",
        cmap="viridis",
        interpolation="nearest",
        extent=[-180, 180, n_units - 0.5, -0.5],
    )
    ax.set_yticks(np.arange(n_units))
    ax.set_yticklabels([
        f"{probe}:{unit_id}" for probe, unit_id in unit_key_sequence
    ])
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_xticklabels(["180", "90", "0", "270", "180"])
    ax.set_xlabel("HD direction (deg; RF spatial layout)")
    ax.set_ylabel("Unit ID")
    fig.colorbar(image, ax=ax, label="Normalized response")
    fig.tight_layout()
    plt.show()
    return fig, ax

In [ ]:
# 1. All RF cells, sorted by RF most-fired-bin direction.
plot_keyed_heatmap(rf_profile_by_key, rf_sort_sequence)

# 2. All HD cells, sorted by HD preferred direction.
plot_keyed_heatmap(hd_rate_by_key, hd_sort_sequence)

In [ ]:
# 3-4. Overlap cells: both modalities use the HD order.
plot_keyed_heatmap(hd_rate_by_key, overlap_hd_sort_sequence)
plot_keyed_heatmap(rf_profile_by_key, overlap_hd_sort_sequence)

In [ ]:
# 5-6. Overlap cells: both modalities use the RF order.
plot_keyed_heatmap(rf_profile_by_key, overlap_rf_sort_sequence)
plot_keyed_heatmap(hd_rate_by_key, overlap_rf_sort_sequence)

In [ ]:
def circular_distance_matrix_deg(angles_deg):
    delta = angles_deg[:, None] - angles_deg[None, :]
    return np.abs((delta + 180.0) % 360.0 - 180.0)


def pairing_permutation_p(values, statistic):
    result = permutation_test(
        (values,),
        statistic,
        permutation_type="pairings",
        vectorized=False,
        n_resamples=statistics_permutations,
        alternative="greater",
        rng=np.random.default_rng(statistics_seed),
    )
    return float(result.pvalue)


def circular_correlation(alpha, beta):
    alpha_centered = np.sin(alpha - circmean(alpha))
    beta_centered = np.sin(beta - circmean(beta))
    return float(pearsonr(alpha_centered, beta_centered).statistic)


def circular_correlation_test(alpha, beta):
    rho = circular_correlation(alpha, beta)
    p_value = pairing_permutation_p(
        beta,
        lambda shuffled: abs(circular_correlation(alpha, shuffled)),
    )
    return rho, p_value


def circular_alignment_test(reference, matched):
    residual_vector = np.mean(np.exp(1j * (matched - reference)))
    p_value = pairing_permutation_p(
        matched,
        lambda shuffled: abs(np.mean(np.exp(1j * (shuffled - reference)))),
    )
    return float(np.angle(residual_vector)), float(abs(residual_vector)), p_value


def spearman_mantel_test(hd_angles_deg, rf_angles_deg):
    upper_triangle = np.triu_indices(hd_angles_deg.size, k=1)
    hd_distances = circular_distance_matrix_deg(hd_angles_deg)[upper_triangle]

    def statistic(shuffled_rf_angles):
        rf_distances = circular_distance_matrix_deg(
            shuffled_rf_angles
        )[upper_triangle]
        return float(spearmanr(hd_distances, rf_distances).statistic)

    rho = statistic(rf_angles_deg)
    p_value = pairing_permutation_p(
        rf_angles_deg,
        lambda shuffled: abs(statistic(shuffled)),
    )
    return rho, p_value

## Same-unit and pairwise analyses

In [ ]:
overlap_hd_peak_angles_deg = np.asarray([
    hd_peak_angle_by_key[key] for key in overlap_unit_keys
])
overlap_rf_peak_angles_deg = np.asarray([
    rf_peak_angle_by_key[key] for key in overlap_unit_keys
])
overlap_hd_peak_angles_rad = np.deg2rad(overlap_hd_peak_angles_deg)
overlap_rf_peak_angles_rad = np.deg2rad(overlap_rf_peak_angles_deg)

rho_circ, p_circ = circular_correlation_test(
    overlap_hd_peak_angles_rad,
    overlap_rf_peak_angles_rad,
)
alignment_offset_rad, alignment_resultant, p_alignment = circular_alignment_test(
    overlap_hd_peak_angles_rad,
    overlap_rf_peak_angles_rad,
)
alignment_offset_deg = float(np.rad2deg(alignment_offset_rad))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    overlap_hd_peak_angles_deg,
    overlap_rf_peak_angles_deg,
    s=28,
    alpha=0.8,
)
ax.set(
    xlabel="HD preferred direction (deg)",
    ylabel="RF most-fired-bin direction (deg)",
    title="Same-unit HD and RF peak directions",
    xlim=(0, 360),
    ylim=(0, 360),
    xticks=np.arange(0, 361, 90),
    yticks=np.arange(0, 361, 90),
)
ax.text(
    0.03,
    0.97,
    (
        f"Circular-circular rho = {rho_circ:.3f}\n"
        f"p = {p_circ:.4g}\n"
        f"1:1 offset = {alignment_offset_deg:+.1f} deg; "
        f"R = {alignment_resultant:.3f}"
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.9},
)
ax.set_aspect("equal", adjustable="box")
ax.grid(color="0.85", linewidth=0.8)
fig.tight_layout()
plt.show()

same_unit_statistics = {
    "circular_correlation_rho": rho_circ,
    "permutation_p": p_circ,
    "one_to_one_offset_deg": alignment_offset_deg,
    "alignment_resultant": alignment_resultant,
    "alignment_permutation_p": p_alignment,
    "unit_count": len(overlap_unit_keys),
}
same_unit_statistics

In [ ]:
mantel_rho, mantel_p = spearman_mantel_test(
    overlap_hd_peak_angles_deg,
    overlap_rf_peak_angles_deg,
)
hd_pair_distances_deg = circular_distance_matrix_deg(
    overlap_hd_peak_angles_deg
).ravel()
rf_pair_distances_deg = circular_distance_matrix_deg(
    overlap_rf_peak_angles_deg
).ravel()
ordered_pair_count = hd_pair_distances_deg.size

density_bin_width_deg = 180.0 / (density_bin_count - 1)
density_bin_edges = np.linspace(
    -density_bin_width_deg / 2.0,
    180.0 + density_bin_width_deg / 2.0,
    density_bin_count + 1,
)
fig, ax = plt.subplots(figsize=(8, 8))
density_image = ax.hist2d(
    hd_pair_distances_deg,
    rf_pair_distances_deg,
    bins=(density_bin_edges, density_bin_edges),
    cmap="viridis",
    norm=PowerNorm(gamma=0.5, vmin=0),
)[3]
fig.colorbar(density_image, ax=ax, pad=0.02)
ax.plot(
    [0, 180],
    [0, 180],
    linestyle="--",
    linewidth=1.2,
    color="white",
    label="Equal pair distance",
)
ax.legend(loc="lower right")
ax.set(
    xlabel="Pairwise HD preferred-direction distance (deg)",
    ylabel="Pairwise RF peak-direction distance (deg)",
    title=f"Pairwise HD and RF peak-distance density ({ordered_pair_count} ordered pairs)",
    xlim=(-5, 185),
    ylim=(-5, 185),
    xticks=np.arange(0, 181, 45),
    yticks=np.arange(0, 181, 45),
)
ax.text(
    0.03,
    0.97,
    (
        f"Spearman Mantel rho = {mantel_rho:.3f}\n"
        f"p = {mantel_p:.4g}\n"
        f"n = {len(overlap_unit_keys)} units ({ordered_pair_count} ordered pairs)"
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    color="black",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.9},
)
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()
plt.show()

pairwise_statistics = {
    "spearman_mantel_rho": mantel_rho,
    "permutation_p": mantel_p,
    "unit_count": len(overlap_unit_keys),
    "ordered_pair_count": ordered_pair_count,
}
pairwise_statistics